In [ ]:
# [목적] 필요한 라이브러리와 환경을 준비합니다.
# [핵심] Pydantic은 출력 형식을 정의하고 LangChain은 모델 호출을 연결합니다.
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

llm = ChatOpenAI(temperature=0, model_name="gpt-5-nano")

In [ ]:
# 이메일 예시
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [ ]:
# [목적] 질문과 이메일을 프롬프트로 만들고 언어 모델과 연결합니다.
# [핵심] {email_conversation}은 실행 시 실제 입력값으로 바뀝니다.
from itertools import chain
from langchain_core.prompts import PromptTemplate

# [목적] 질문, 이메일, 출력 형식을 포함한 최종 프롬프트 틀을 만듭니다.
# [주의] 중괄호 변수명은 실행할 때 전달하는 딕셔너리 키와 같아야 합니다.
prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}" 
)

llm = ChatOpenAI(temperature=0, model_name="gpt-5-nano")
# [목적] 완성된 프롬프트로 모델을 호출해 이메일 요약을 스트리밍합니다.
# [주의] question, email_conversation 키는 프롬프트 변수명과 같아야 합니다.
# [활용] stream은 긴 응답을 조금씩 화면에 보여줄 때 유용합니다.
# [목적] 완성된 프롬프트와 모델을 연결해 구조화 응답을 요청합니다.
# [주의] 입력 딕셔너리의 키는 question, email_conversation과 같아야 합니다.
# [활용] stream은 응답을 조금씩 화면에 보여줄 때 사용합니다.
chain = prompt | llm
answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

In [ ]:
# [목적] 모델이 반환할 이메일 요약의 구조와 자료형을 정의합니다.
# [핵심] 필드가 정해져 결과를 일정하게 검증할 수 있습니다.
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [ ]:
# [목적] EmailSummary 형식으로 모델 응답을 검사하는 파서를 만듭니다.
# [활용] 문자열 응답을 검증된 Python 객체로 바꿀 때 사용합니다.
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [ ]:
# [목적] 모델에게 안내할 JSON 출력 규칙을 확인합니다.
# [핵심] 이 지침을 프롬프트에 넣으면 일정한 형식의 응답을 유도합니다.
print(parser.get_format_instructions())

In [ ]:
# [목적] 질문, 이메일, 출력 형식을 포함한 최종 프롬프트 틀을 만듭니다.
# [주의] 중괄호 변수명은 실행 시 전달하는 키와 정확히 같아야 합니다.
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following the questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{Format}
"""
)

In [ ]:
# [목적] 출력 형식 지침을 프롬프트에 미리 채워 넣습니다.
# [주의] {Format}과 partial의 Format은 대소문자까지 일치해야 합니다.
prompt = prompt.partial(Format=parser.get_format_instructions())
prompt

In [ ]:
chain = prompt | llm

# [목적] 완성된 프롬프트와 모델을 연결해 구조화 응답을 요청합니다.
# [주의] 입력 키는 question, email_conversation과 같아야 합니다.
# [활용] stream은 응답을 조금씩 화면에 보여줄 때 사용합니다.
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

In [ ]:
# [목적] 모델의 JSON 문자열을 EmailSummary 객체로 변환하고 검증합니다.
# [핵심] 필드가 누락되거나 형식이 다르면 여기서 오류가 발생합니다.
structured_output = parser.parse(output)
print(structured_output)

In [ ]:
# [목적] 구조화된 결과에서 필요한 필드만 꺼내 쓰는 예시입니다.
# [활용] 데이터베이스 저장, 화면 표시, 후속 자동화에 활용할 수 있습니다.
structured_output.person

In [ ]:
# [목적] 프롬프트 → 모델 → 파서를 한 줄 체인으로 연결합니다.
# [핵심] invoke는 전체 결과를 한 번에 반환하므로 객체 처리에 편리합니다.
# [비교] stream은 실시간 출력, invoke는 최종 결과 처리에 적합합니다.
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요."
    }
)

response